In [6]:
!pip install finnhub-python
!pip install apache-airflow-providers-databricks
!pip install requests
!pip install apache-airflow-providers-google
!pip install apache-airflow-providers-slack
!pip install dockers












  Using cached oauthlib-3.3.1-py3-none-any.whl.metadata (7.9 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Using cached oauthlib-3.3.1-py3-none-any.whl (160 kB)
  Created wheel for thrift: filename=thrift-0.20.0-py3-none-any.whl size=196263 sha256=e48c121d05814ff1ad4cbfae0222c963737f2b3d9d274038926fa3243a73493e
  Stored in directory: c:\users\shiim\appdata\local\pip\cache\wheels\57\4d\9e\539ef8e052f8e2db244ddf8ac1f2e623b4663f2ee233ffafb1
Successfully built thrift

   ---------------------------------------- 0/8 [thrift]
   ---------- ----------------------------- 2/8 [oauthlib]
   ---------- ----------------------------- 2/8 [oauthlib]
   -------------------- ------------------- 4/8 [databricks-sql-conn

ERROR: Could not find a version that satisfies the requirement dockers (from versions: none)
ERROR: No matching distribution found for dockers


In [0]:

%restart_python

In [0]:

import logging
from tenacity import retry, stop_after_attempt, wait_exponential_jitter,retry_if_exception_type,before_sleep_log
from urllib3.util.retry  import Retry
import requests
from airflow.sdk.bases.hook import BaseHook
from requests.adapters import HTTPAdapter
from airflow.exceptions import AirflowException
from typing import List,Dict,Any
import time



logger =logging.getLogger(__name__)



class FinnhubAPI(BaseHook):
    conn_name_attr = "api_conn_id"
    default_conn_name = "finnhub_default"
    conn_type = "http"
    hook_name = "Finnhub"
    base_url = "https://api.finnhub.io/api/v1"
    default_timeout = 10
    


    def __init__(
        self,
        api_conn_id:str = default_conn_name,
        base_url:str = base_url,
        max_retries:int = 3,
        timeout:int = default_timeout,
        rate_limit_per_min:int = 60,
        ):
        super().__init__()
        self.api_conn_id = api_conn_id
        self.base_url = base_url
        self.max_retries = max_retries
        self.timeout = timeout
        self.rate_limit_per_min = rate_limit_per_min
        self._session = None
        

    def get_api_key(self)->str:

        """The Api_key is provided directly in the Finnhub Documentation.
        But i have chosen the get_connection() functionality to add a security layer.
        Though,the Api key will be displayed on the Apache Airflow UI Dashboard"""

        conn = self.get_connection(self.api_conn_id)
        if not conn.password:
            raise AirflowException(
                f"No API key found for the connection. Set the correct API Key")
        return conn.password         
        
        
    def _conn_session(self)-> requests.Session:
        if self._session is None:
            self._session = requests.Session()
            self._session.headers.update({"Accept": "application/json",
                                "User-Agent": "finnhub/python"})
            self._session.params["token"] = self.get_api_key()

            retry_attributes = Retry(
                total = self.max_retries,
                backoff_factor = 3,
                status_forcelist= [429,500,502,503,504],
                allowed_methods = ["GET"]
            )
            adapter = HTTPAdapter(max_retries = retry_attributes)
            self._session.mount("https://",adapter)
            self._session.mount("http://",adapter)
        return self._session



    @retry(
        retry = retry_if_exception_type(
            (requests.exceptions.HTTPError, ValueError,requests.exceptions.ConnectionError,requests.exceptions.Timeout)),
        wait = wait_exponential_jitter(initial =1, max=60, jitter =3),
        stop= stop_after_attempt(3),
        before_sleep = before_sleep_log(logger, logging.WARNING),
        reraise = True,
    )
    def fetch_data(self, name:str)-> Dict[str,Any]:
        session = self._conn_session()
        parameters = {
           "symbol": name
        }
        web_url = f"{self.base_url}/quote"

        logger.info("Fetching Financial Data!")
        response = session.get(web_url , params = parameters, timeout = self.timeout)
        if response.status_code == 429:
            logger.warning(f"Rate Limit Hit for {name}! Tenacity will retry...")
            response.raise_for_status() 

        try:
            response.raise_for_status()
            logger.info("Successfully fetched the data!")
            return response.json()
            
        except requests.exceptions.HTTPError as ERROR:
            if ERROR.response.status_code == 401:
                # Fatal error: Don't retry, just fail the Airflow task
                raise AirflowException("Invalid API Key! Check Your Airflow Connection.")
            elif ERROR.response.status_code == 404:
                # Missing data: Don't fail the task, just skip this symbol
                logger.warning(f"Symbol {name} not found.")
                return None
            raise
                
                
        

    def fetch_data_in_batches(self, names: List[str]) -> Dict[str, Any]:
        result = {}
        delay = 60.00 / self.rate_limit_per_min

        for i, name in enumerate(names):
            # 1. Apply delay BEFORE the call (except for the first one)
            if i > 0:
                time.sleep(delay)
            
            # 2. Call the API once
            data = self.fetch_data(name)
            
            # 3. Store the result in the dictionary
            result[name] = data
        
    # Filter out the Nones (404s) if you only want successful records
        successful_results = {k: v for k, v in result.items() if v is not None}
        
        logger.info(f"Fetched {len(successful_results)}/{len(names)} symbols successfully.")
        return successful_results
        
    def test_connection(self):
        try:
            aapl_data = self.fetch_data("AAPL")
            print(f"AAPL Data: {aapl_data}")
            return True, "Connection Successful!"
        except Exception as e:
            return False, f"Connection failed: {str(e)}"
        
    
        
    def close(self):
        if self._session:
            self._session.close()
            self._session = None

           
        
            
        
        

        




        




In [0]:
import logging 
from dataclasses import dataclass
from typing import Tuple,List
import pandas as pd

logger = logging.getLogger(__name__)

@dataclass
class FinnhubValidationResult:
    check_name: str
    passed: bool
    message: str
    severity: str

class FinnhubDataValidator:
    def __init__(self,df:pd.DataFrame):
        self.df = df

    def validate(self)-> Tuple[bool,List[FinnhubValidationResult]]:
        results = []
        results.append(self._check_not_empty(self.df))
        results.append(self._check_columns(self.df))
        results.append(self._data_types(self.df))
        results.append(self._check_null(self.df))
        results.append(self._duplicate_values(self.df))
        for result in results:
            if result.passed:
                logger.info(f"{result.check_name}:{result.message}")
            elif result.severity == "warning":
                logger.warning(f"{result.check_name}:{result.message}")
            else:
                logger.error(f"{result.check_name}:{result.message}")
        critical_failures = [ result for result in results if not result.passed and result.severity == "critical"]
        all_passed = len(critical_failures) == 0

        if not all_passed:
            failure_messages = [failure.message for failure in critical_failures]
            logger.error(f"Validation Failed:{failure_messages}")
        return all_passed,results

    def _check_not_empty(self, df:pd.DataFrame)-> FinnhubValidationResult:
        passed = len(df) > 0
        return FinnhubValidationResult(
            check_name = "Not Empty",
            passed = passed,
            message = f"Dataframe is not Empty and has length {len(df)}" if passed  else "Dataframe is Empty",
            severity = "critical"
        )

    def _check_columns(self, df:pd.DataFrame)-> FinnhubValidationResult:
        available_columns = {"c","d","dp","h","l","o","pc"}
        missing_columns = available_columns - set(df.columns)
        passed = len(missing_columns) == 0
        return FinnhubValidationResult(
            check_name = "Not Missing",
            passed = passed,
            message = "All Columns present" if passed  else f"The Dateframe has {missing_columns} Columns Missing.",
            severity = "critical"
        )
    
    def _data_types(self, df:pd.DataFrame)-> FinnhubValidationResult:
        data_types = {"c":float,"d":float,"dp":float,"h":float,"l":float,"o":float,"pc":float}
        for k,v in data_types.items():
            if k not in df.columns:
                continue
            if df[k].dtype != v:
                return FinnhubValidationResult(
                    check_name = "Data Types",
                    passed = False,
                    message = f" Column {k} is not of the Float Data Type",
                    severity = "critical"
                )
        return FinnhubValidationResult(
        check_name="Data Types",
        passed=True,
        message="All columns have correct data types",
        severity="critical",
    )
               
    def _check_null(self, df:pd.DataFrame) -> FinnhubValidationResult:
        null_counts = df.isnull().sum()
        total_nulls = null_counts.sum()
        cols_with_nulls = null_counts[null_counts > 0].to_dict()
        passed = total_nulls == 0
        return FinnhubValidationResult(
            check_name = "Null Values",
            passed = passed,
            message = "No null values" if passed else f"Null values: {cols_with_nulls}",
            severity = "warning"
        )

    def _duplicate_values(self, df:pd.DataFrame) -> FinnhubValidationResult:
        duplicate_values = df.duplicated().sum()
        passed = duplicate_values == 0
        return FinnhubValidationResult(
            check_name = "Duplicate Values",
            passed = passed,
            message = f"DataFrame has {duplicate_values} duplicate values",
            severity = "warning"
        )



In [0]:
import pytest
import requests
import pandas as pd
from airflow.exceptions import AirflowException
from unittest.mock import MagicMock,patch


SAMPLE_QUOTE = pd.DataFrame([{
    "c": 250.10, "d": 2.50, "dp": 1.01,
    "h": 252.00, "l": 248.50, "o": 249.00,
    "pc": 247.60
}])
EMPTY_QUOTE = ([{
    "c": 0, "d": None, "dp": None,
    "h": 0, "l": 0, "o": 0, "pc": 0
}])

def test_session(client,mocker):
    mock_session = MagicMock()
    mocker.patch("my_pipeline_requests.session",return_value = mock_session)
    FinnhubAPI.fetch_data("AAPL")
    mock_session.get.assert_called()

class TestValidators:
    def test_check_not_empty_passes(self):
        result = FinnhubDataValidator(SAMPLE_QUOTE)._check_not_empty(SAMPLE_QUOTE)
        assert result.passed is True
        assert "length" in result.message

    def test_check_not_empty_fails(self):
        result = FinnhubDataValidator(EMPTY_QUOTE)._check_not_empty(EMPTY_QUOTE)
        assert result.passed is False
        assert result.severity == "critical"

    def test_check_columns_passes(self):
        result = FinnhubDataValidator(SAMPLE_QUOTE)._check_columns(SAMPLE_QUOTE)
        assert result.passed is True
    
    def test_check_columns_fails(self):
        bad_df = SAMPLE_QUOTE.drop(columns = ["c"])
        result = FinnhubDataValidator(bad_df)._check_columns(bad_df)
        assert result.passed is False
        assert "Missing" in result.message

    def test_duplicate_values_passes(self):
        result = FinnhubDataValidator(SAMPLE_QUOTE)._duplicate_values(SAMPLE_QUOTE)
        assert result.passed is True
    
    def test_duplicate_values_fails(self):
        validator = FinnhubDataValidator(EMPTY_QUOTE)
        passed,results = validator.validate()
        assert passed is False
        
    def test_validate_overall_critical_fail(self):
        null_df = SAMPLE_QUOTE.copy()
        null_df.loc[0,"c"] = None
        validator = FinnhubDataValidator(null_df)
        passed,results = validator.validate()
        assert passed is False
        assert len(results) == 5
        

    def test_validate_overall_warnings(self):
        null_df = SAMPLE_QUOTE.copy()
        null_df.loc[0,"c"] = None
        validator = FinnhubDataValidator(null_df)
        passed,results = validator.validate()
        assert passed is True
        

class TestFetchData:
    @patch("my_pipeline_requests.session")
    def test_no_connection_error(self,mock_session,client):
        mock_resp = MagicMock()
        mock_resp.json.return_value = {"c": 250.10}
        mock_resp.raise_for_status.return_value = None
        mock_resp.return_value.get.return_value = mock_resp
        result = FinnhubAPI.fetch_data("AAPL")
        assert result["c"] == 250.10

    @patch("my_pipeline_request.session")
    def test_return_none_404_error(self,mock_session,client):
        mock_resp = MagicMock()
        mock_resp.raise_for_status.side_effect = requests.HTTPError("404")
        mock_resp.status_code = 404
        mock_session.return_value.get.return_value = mock_resp
        result = client.fetch_data("BAD")
        assert result is None




In [0]:
from dataclasses import dataclass, field
from typing import List

@dataclass(frozen = True)
class APIConfigurations:
    base_url:str = "https://api.finnhub.io/api/v1"
    api_conn_id:str = "finnhub_default"
    rate_limit_per_min:int = 60
    max_retries:int = 3
    default_timeout:int = 10

@dataclass(frozen = True)
class GCSConfig:
    bucket_name:str = "blob_finnhub"
    blob_path:str = "blob_finnhub/finnhub_etl"
    raw_prefix:str = "raw"
    transformed_prefix:str = "transformed"
    file_format:str = "parquet"

@dataclass(frozen = True)
class BigQueryConfig:
    project:str = "project-ba01e344-5e1a-4b59-8d5"
    dataset:str = "finnhub"


@dataclass(frozen=True)
class AlertConfig:
    slack_conn_id:str = "slack_webhook"
    slack_channel:str = "Data-Pipeline-Alert"
    alert_on_retry:bool = False
    alert_on_failure:bool = True

@dataclass(frozen=True)
class PipelineConfig:
    api:APIConfigurations = field(default_factory=APIConfigurations)
    gcs:GCSConfig = field(default_factory = GCSConfig)
    bigquery:BigQueryConfig = field(default_factory=BigQueryConfig)
    alert:AlertConfig = field(default_factory=AlertConfig)
    gcp_conn_id: str = "google_cloud_default"

pipeline_config = PipelineConfig()


In [0]:
from datetime import datetime, timedelta
from typing import Dict, Any

import pandas as pd
import io
import json
import logging
import sys

from airflow import DAG
from airflow.decorators import task
from airflow.providers.google.cloud.hooks.bigquery import BigQueryHook
from airflow.providers.slack.notifications.slack import send_slack_notification
from airflow.exceptions import AirflowException
from google.cloud import storage


sys.path.insert(0, "/opt/airflow")
logger = logging.getLogger(__name__)

def _write_parquet_to_gcs(df:pd.DataFrame, bucket_name:str, blob_path:str,project:str)-> str:
    client = storage.Client(project = project)
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_path)
    buffer = io.BytesIO()
    df.to_parquet(buffer, index = False, engine = "pyarrow")
    buffer.seek(0)
    blob.upload_from_file(buffer,content_type = "application/octet-stream")
    return f"gs://{bucket_name}/{blob_path}"

def _read_parquet_from_gcs(project:str,bucket_name:str,blob_path:str)-> pd.DataFrame:
    client = storage.Client(project = "project-ba01e344-5e1a-4b59-8d5")
    bucket = client.bucket("blob_finnhub")
    blob = bucket.blob("blob_finnhub/finnhub_etl")
    buffer = io.BytesIO()
    blob.download_to_file(buffer)
    buffer.seek(0)
    return pd.read_parquet(buffer, engine = "pyarrow")

def failure_callback(context:Dict[str, Any]):
    task_instance_failed = context["task_instance"]
    exception = context.get("exception","Unknown error")
    logical_date = context.get("logical_date","Unknown date")

    message = (
        f"Pipeline Failure\n"
        f"DAG: {task_instance_failed.dag_id}\n"
        f"Task: {task_instance_failed.task_id}\n"
        f"Date: {logical_date}\n"
        f"Error:{str(exception)[:500]}\n"
        f"Log: {task_instance_failed.log_url}"
    )
    logger.error(message)

default_args = {
    "owner":"finnhub_etl",
    "depends_on_past":False,
    "retries":2,
    "retry_delay": timedelta(minutes = 5),
    "retry_exponential_backoff": True,
    "on_failure_callback": failure_callback,
    "execution_timeout": timedelta(minutes = 10),
    }
with DAG(
    dag_id = "Finnhub_ETL",
    default_args = default_args,
    start_date = datetime(2026,1,1),
    schedule = "@daily",
    catchup = False,
    max_active_runs = 1,
    tags = ["Finnhub","ETL"],
    doc_md = __doc__,
) as dag:
    @task(
        retries = 3,
        retry_delay = timedelta(minutes = 2),
    )

    def extract(logical_date = None, **kwargs)->str:
        hook = FinnhubAPI(
            api_conn_id = pipeline_config.api.api_conn_id,
            default_timeout = pipeline_config.api.default_timeout,
            max_retries = pipeline_config.api.max_retries,
            rate_limit_per_min = pipeline_config.api.rate_limit_per_min
        )
        hook_results = hook.fetch_data_in_batches(list(pipeline_config.api.default_symbols))
        if not hook_results:
            raise AirflowException("No data returned. Got an Empty Response in the form of an empty List")

        records = []
        for result in hook_results:
            records.append({
                "symbol": result["symbol"],
                "c": result["c"],
                "t": result["t"],
                "o": result["o"],
                "h": result["h"],
                "l": result["l"],
                "v": result["v"],
                "d": logical_date.strftime("%Y-%m-%d"),
                "dp": result["dp"],
                "pc": result["pc"]
            })
        df = pd.DataFrame(records)

        date_partition = logical_date.strftime("%Y-%m-%d")
        gcs_path = f"{pipeline_config.gcs.raw_prefix}/date={date_partition}/finnhub_data.parquet"
        _write_parquet_to_gcs(df, pipeline_config.gcs.bucket_name, gcs_path, project)
        logger.info(f"Extracted {len(df)} records → gs://{pipeline_config.gcs.bucket_name}/{gcs_path}")
        return gcs_path
    
    @task
    def validate(raw_gcs_path:str, **kwargs) -> str:
        bigquery_hook = BigQueryHook(gcp_conn_id = pipeline_config.gcp_conn_id)
        project = bigquery_hook.get_client().project
        df = _read_parquet_from_gcs(pipeline_config.gcs.bucket,raw_gcs_path,project)
        passed,results = Validator.validate_raw_data(df)

        if not passed:
            fail = [r for r in results if not r.passed and r.severity == "critical"]
            fail_info = ",".join([f"{r.check_name}:{r.message}"for r in fail])
            raise AirflowException(f"Data Validation Failed:{fail_info}")
        logger.info("All Validation Checks Cleared!")
        return raw_gcs_path

    @task
    def transform(validated_gcs_path:str, logical_date = None, **kwargs) -> Dict[str,str]:
        bigquery_hook = BigQueryHook(gcp_conn_id = pipeline_config.gcp_conn_id)
        project = bigquery_hook.get_client().project
        df = _read_parquet_from_gcs(pipeline_config.gcs.bucket_name,validated_gcs_path,project)
        
        date_key = int(logical_date.strftime("%Y%m%d"))

        dim_symbol = (
        df[["symbol"]]
        .drop_duplicates()             
        .reset_index(drop=True)
        )

        dim_date = pd.DataFrame([{
            "date_key": date_key,
            "full_date": str(date_key),
            "day_name": logical_date.strftime("%A"),              
            "day_of_week": logical_date.weekday(),               
            "day_of_month": logical_date.day,                     
            "week_of_year": logical_date.isocalendar()[1],       
            "month_name": logical_date.strftime("%B"),            
            "month_number": logical_date.month,                   
            "quarter": f"Q{(logical_date.month - 1) // 3 + 1}", 
            "year": logical_date.year,                            
            "is_weekend": logical_date.weekday() >= 5,            
            "is_month_start": logical_date.day == 1,              
            "is_month_end": (                                     
                logical_date.month != (logical_date + pd.Timedelta(days=1)).month
            ),
        }])

        fact_df = df.merge(
        dim_symbol[["ticker", "symbol_key"]],
        left_on="symbol",
        right_on="ticker",
        )


        fact_df["date_key"] = date_key

        fact_df = fact_df.rename(columns={
            "c": "close_price",
            "o": "open_price",
            "h": "high_price",
            "l": "low_price",
            "pc": "previous_close",
            "d": "price_change",
            "dp": "price_change_percent",
        })
        fact_columns = [
            "date_key",
            "symbol_key",
            "close_price",
            "open_price",
            "high_price",
            "low_price",
            "previous_close",
            "price_change",
            "price_change_percent",
        ]
        available_columns = [col for col in fact_columns if col in fact_df.columns]
        fact_df = fact_df[available_columns]
        fact_df["ingestion_date"] = str(date_key)
        tables = {
            "dim_date": dim_date,
            "dim_symbol": dim_symbol,
            "fact_daily_quotes": fact_df,
        }

        output_paths = {}
        for table_name, table_df in tables.items():
            path = f"{pipeline_config.gcs.transformed_prefix}/date={str(date_key)}/{table_name}.parquet"
            _write_parquet_to_gcs(table_df, pipeline_config.gcs.bucket, path, project)
            output_paths[table_name] = path
            logger.info(f"Wrote {table_name} ({len(table_df)} rows) → {path}")

        return output_paths

    @task
    def load(table_paths: Dict[str, str], **kwargs) -> None:
        
        from google.cloud.bigquery import LoadJobConfig, WriteDisposition

        bq_hook = BigQueryHook(gcp_conn_id=pipeline_config.gcp_conn_id)
        client = bq_hook.get_client()
        dataset = pipeline_config.bigquery.dataset

        write_modes = {
            "dim_date": WriteDisposition.WRITE_TRUNCATE,
            "dim_symbol": WriteDisposition.WRITE_TRUNCATE,
            "fact_daily_quotes": WriteDisposition.WRITE_APPEND,
        }

        for table_name, gcs_path in table_paths.items():
            df = _read_parquet_from_gcs(pipeline_config.gcs.bucket, gcs_path, client.project)

            table_id = f"{pipeline_config.bigquery.project}.{dataset}.{table_name}"
            disposition = write_modes.get(table_name, WriteDisposition.WRITE_APPEND)

            job_config = LoadJobConfig(write_disposition=disposition)
            job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
            job.result()  # Wait for completion

            logger.info(
                f"Loaded {table_name} → {table_id} "
                f"({len(df)} rows, mode={disposition})"
            )
    raw_path = extract()
    validated_path = validate(raw_path)
    transformed_paths = transform(validated_path)
    load(transformed_paths)

2026-04-16T19:42:49.536109Z [warning  ] The `airflow.decorators.task` attribute is deprecated. Please use `'airflow.sdk.task'`. [py.warnings] category=DeprecatedImportWarning filename=/home/spark-d127b65e-2191-4c04-b248-a4/.ipykernel/3576/command-7229885747719906-2967753366 lineno=11


In [ ]:
!cd C:\Users\shiim\Downloads\Docker_Finnhub



In [5]:
!astro dev init

^C
